# 03 — 10 stocks, SB3 SAC

Same protocol as notebook 01: configured dataset + universe, YAML dates, custom `(f,n,t)` extractor. Default `MlpPolicy`/`CnnPolicy` are not used.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sp500rl.config import load_config
from sp500rl.seed import set_seed

CFG = load_config(ROOT / "configs" / "default.yaml")
SEED = set_seed(int(CFG["seed"]))
DATASET = CFG["dataset"]            # configs/datasets.yaml entry (synthetic | ab_wrds | yahoo | ...)
UNIVERSE = CFG["universe"]["rule"]  # ab_finrl | sandbox | full_window | top_n | listed
print(f"seed={SEED} dataset={DATASET} universe={UNIVERSE}")
print("train", CFG["dates"]["train_start"], "→", CFG["dates"]["train_end"])
print("test ", CFG["dates"]["test_start"], "→", CFG["dates"]["test_end"])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, SAC

from sp500rl.data import get_panel
from sp500rl.env.extractors import sb3_policy_kwargs
from sp500rl.env.make_env import make_poe
from sp500rl.baselines.simple import rollout, rollout_equal_weight, rollout_risk_parity
from sp500rl.eval.metrics import compare_rollouts

panel = get_panel(DATASET, UNIVERSE, CFG)
print("panel", panel.shape, "tickers", sorted(panel["tic"].unique()))


In [ ]:
ALGO = "SAC"
print(f"SB3 {ALGO} seed={SEED}")

train_env = make_poe(
    panel, CFG, mode="train",
    wrap_gymnasium=True, return_last_action=True, new_gym_api=True,
    cwd=ROOT / "results" / f"sb3_{ALGO.lower()}_train",
)
obs, info = train_env.reset(seed=SEED)
print("obs keys", obs.keys(), "state", obs["state"].shape, "last_action", obs["last_action"].shape)

Algo = PPO if ALGO == "PPO" else SAC
kwargs = dict(
    policy="MultiInputPolicy",
    env=train_env,
    seed=SEED,
    verbose=0,
    policy_kwargs=sb3_policy_kwargs(64),
)
if ALGO == "PPO":
    kwargs.update(n_steps=256, batch_size=64, n_epochs=4)
else:
    kwargs.update(buffer_size=5_000, batch_size=64, learning_starts=256, train_freq=64)

model = Algo(**kwargs)
TIMESTEPS = 2048  # bump for research runs
print("learning", TIMESTEPS, "timesteps")
model.learn(total_timesteps=TIMESTEPS)


In [ ]:
def rollout_sb3(model, env):
    def fn(obs, info, t):
        action, _ = model.predict(obs, deterministic=True)
        return np.asarray(action, dtype=float)
    return rollout(env, fn)

test_kw = dict(mode="test", wrap_gymnasium=True, return_last_action=True, new_gym_api=True)
agent_env = make_poe(panel, CFG, cwd=ROOT / "results" / f"sb3_{ALGO.lower()}_agent", **test_kw)
eq_env = make_poe(panel, CFG, cwd=ROOT / "results" / f"sb3_{ALGO.lower()}_eq", **test_kw)
rp_env = make_poe(panel, CFG, cwd=ROOT / "results" / f"sb3_{ALGO.lower()}_rp", **test_kw)

agent_roll = rollout_sb3(model, agent_env)
eq_roll = rollout_equal_weight(eq_env)
rp_roll = rollout_risk_parity(rp_env, panel)
summary = compare_rollouts({ALGO: agent_roll, "equal_weight": eq_roll, "risk_parity": rp_roll})
print(summary.to_string())

def _series(roll, name):
    n = min(len(roll["dates"]), len(roll["values"]))
    return pd.Series(roll["values"][:n], index=pd.to_datetime(roll["dates"][:n]), name=name)

fig, ax = plt.subplots(figsize=(10, 4))
for roll, name in ((agent_roll, ALGO), (eq_roll, "equal-weight"), (rp_roll, "risk-parity")):
    _series(roll, name).plot(ax=ax)
ax.set_title("Test-window portfolio value")
ax.set_ylabel("value")
plt.tight_layout()
plt.show()
